# Loading ERA5 Data

> *Pre-requisites*: Code requires most of the packages listed [here](https://github.com/google-research/arco-era5/tree/main/docs/environment.yml):

We will open the Zarr data with XArray after getting GCS permissions. We can test bucket access with fsspec:

# The goal of this notebook:

Load the NOAA event data and the NWS county data

Identify which counties are associated with each weather event

Creating a new dataframe in which each row is a county's experience of a weather event

Identifying the time when the weather event entered and left the county (assuming constant velocity of the weather event)

Computing the duration of the event in each county

Looking up the ERA5 weather data closest to the... mean time of the event?

But we also need to keep things in context of the overall severity of the event. Maybe also compute total duration for each event, and find some way to measure total severity from ERA5 data?

(note: this isn't the same as loading all the ERA5 data and merging that with outages... it's just augmenting NOAA instead)

First, load the NOAA data

In [1]:
import pandas as pd

df_events = pd.read_csv("../Data/NOAA_StormEvents/StormEvents_2014_2024.csv")

Next, load the US Counties shapefile

In [2]:
import geopandas

#Load the NWS US Counties shapefile
counties = geopandas.read_file('../Data/NWS_US_Counties_Shapefiles')

In [3]:
#Make a new dataframe called county_fips that consists of just the COUNTYNAME and FIPS variables from counties
county_fips = counties[['COUNTYNAME', 'FIPS']]

###WE NEED TO INCLUDE THE STATE IN THE DATA SET - IN THE NOAA DATA ONE COUNTY NAME MIGHT BE COMMON TO MULTIPLE STATES...

#Convert FIPS into a float
county_fips['FIPS'] = county_fips['FIPS'].astype(float)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_44626/1582361533.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  county_fips['FIPS'] = county_fips['FIPS'].astype(float)


Define a function that identifies all the FIPS in the path of the weather event.

We're going to assume that the path of the event is essentially lineary. This is probably inaccurate, but we don't have any additional data (without doing something really clever with the ERA5 data) as an alternative

In [4]:
#Given a beginning point (given by BEGIN_LAT and BEGIN_LON) and an ending point (given by END_LAT and END_LON) from df_events, identify which FIPS values from counties lie in the path between the beginning and ending points
def get_fips_from_path(row):
    #Get the beginning and ending points
    begin = (row['BEGIN_LON'], row['BEGIN_LAT'])
    end = (row['END_LON'], row['END_LAT'])
    
    points = geopandas.points_from_xy([begin[0], end[0]], [begin[1], end[1]])

    #Get the path between the two points
    path = geopandas.GeoSeries(points)
    
    #Get the FIPS values from the counties shapefile that intersect with the path
    fips = counties[counties.geometry.intersects(path.union_all())]['FIPS'].tolist()
    
    return fips

#Apply get_fips_from_path to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Path'] = df_events.apply(get_fips_from_path, axis=1)


In [42]:
#For each row in df_events if the EVENT_NARRATIVE variable includes any of the values in the COUNTYNAME variable in county_fips, record the value of FIPS as part of a list in a new variable called FIPS_from_Narrative
def get_fips_from_narrative(row):
    fips = []
    for county in county_fips['COUNTYNAME']:
        if county in row['EVENT_NARRATIVE']: #row['EVENT_NARRATIVE'].contains(county):
            fips.append(county_fips[county_fips['COUNTYNAME'] == county]['FIPS'].values[0])
    return fips

df_events['EVENT_NARRATIVE'] = df_events['EVENT_NARRATIVE'].str.lower()
county_fips['COUNTYNAME'] = county_fips['COUNTYNAME'].str.lower()

#Apply get_fips_from_narrative to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Narrative'] = df_events.apply(get_fips_from_narrative, axis=1)



#Identify all rows of df_events where EVENT_NARRATIVE includes the text 'County'
#df_events['County_from_Narrative'] = df_events['EVENT_NARRATIVE'].str.contains('county')

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_31360/1800684224.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  county_fips['COUNTYNAME'] = county_fips['COUNTYNAME'].str.lower()


TypeError: argument of type 'float' is not iterable

In [29]:
#Create a new dataframe from df_events where the variable County_from_Narrative is nonempty
df_events_county = df_events[df_events['County_from_Narrative'] == True]

In [1]:
import fsspec

fs = fsspec.filesystem('gs')
fs.ls('gs://weatherbench2/datasets/era5/')

['weatherbench2/datasets/era5/',
 'weatherbench2/datasets/era5/1959-2022-1h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-512x256_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x33.zarr',
 'weatherbench2/datasets/era5/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2',
 'weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg-chu

Next, we'll load the 6-hour downsampled data set

In [41]:
import xarray as xr

reanalysis = xr.open_zarr(
    'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr', 
    chunks={'time': 48},
    consolidated=True,
    decode_timedelta=True    
)

Let's reduce the size by only including data since 1/1/2014 and only our target variables:
- 10m_wind_speed
- 2m_temperature
- snow_depth
- total_precipitation_12hr
- total_precipitation_24hr
- total_precipitation_6hr
- wind_speed

In [ ]:
#Select time values since 1/1/2014 and the following variables: 
features = [
    'latitude', 
    'longitude', 
    'time', 
    '10m_wind_speed', 
    '2m_temperature', 
    'snow_depth', 
    'total_precipitation_12hr', 
    'total_precipitation_24hr', 
    'total_precipitation_6hr', 
]

reanalysis = reanalysis.sel(time=slice('2014', '2021'))[features]

Next, we'll restrict our data to county centroids (in the counties_centroids.csv file created by the convert_NWS_shapefile_to_county_centroids notebook).

In the ERA5 coordinate system, latitude values are "normal" but longitude values are expressed as values within [0, 360] with respect to the Greenwich Prime Meridian (i.e., instead of [-180, 180]). Since our county centroid data are all West of the Prime Meridian, we can simply adjust their longitude values with an auxiliary function.

In [43]:
#Use xarray to load the file ../Data/counties_centroids.csv
import pandas as pd
counties_centroids = pd.read_csv('../Data/counties_centroids.csv')

counties_centroids['LON'] = counties_centroids['LON'].astype(float)
counties_centroids['LAT'] = counties_centroids['LAT'].astype(float)

#The function below converts "standard" longitude values to ERA5 longitude values
def lon_to_360(dlon: float) -> float:
  return ((360 + (dlon % 360)) % 360)

counties_centroids['LON'] = counties_centroids['LON'].apply(lon_to_360)

In [14]:
#Create a new variable 'position' in counties_centroids that is the ordered pair from LON and LAT
counties_centroids['position'] = list(zip(counties_centroids['LON'], counties_centroids['LAT']))

In [44]:
#Create a new DataArray from counties_centroids with the LON and LAT values as longitude and latitude coordinates and FIPS as the data
counties_centroids_da = xr.DataArray(
    counties_centroids['FIPS'].values,
    coords={
        'longitude': ('points', counties_centroids['LON'].values),
        'latitude': ('points', counties_centroids['LAT'].values)
    },
    dims='points'
)

In [45]:
#Restrict the reanalysis data set to coordinates in counties_centroids_da
reanalysis_counties = reanalysis.sel(
    longitude=reanalysis.longitude.isin(counties_centroids_da.longitude),
    latitude=reanalysis.latitude.isin(counties_centroids_da.latitude)
)

We can try saving the zarr file locally as a NetCDF file. The size of the data is roughly 9 GB. However, I've let it run for several hours without it completing.

I've run into roadblocks trying to save locally as a zarr file; I keep getting a "TypeError(f"Expected a BytesBytesCodec. Got {type(data)} instead.")" error. From what I can tell, this appears to be related to zarr 3.

In [14]:
# Compute the size of the reanalysis_counties dataset in GB
#print(f'size: {reanalysis_counties.nbytes / (1024 ** 3)} GiB')

#Export reanalysis_counties to NetCDF
#reanalysis_counties.to_netcdf('../Data/reanalysis_counties.nc')

In [12]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet data set
import pyarrow.parquet as pq
eaglei_outages = pq.read_table('../Data/eaglei_data/eaglei_outages_with_county_info.parquet').to_pandas()

In [15]:
#In eaglei_outages convert centroid to the coordinate pair LON, LAT
eaglei_outages['LON'] = eaglei_outages['centroid'].apply(lambda x: x[0])
eaglei_outages['LAT'] = eaglei_outages['centroid'].apply(lambda x: x[1])